In [3]:
import pandas as pd
import numpy as np
import random
red_line_df = pd.read_pickle("red_line_df.pkl")

# ==========================================
# TASK 1: Clean and Preprocess Data
# ==========================================
# GTFS schedules often use times past midnight (e.g., 25:30:00 for 1:30 AM).
# Standard datetime functions fail here, so we must convert times to total seconds.

def time_to_seconds(time_str):
    if pd.isna(time_str):
        return np.nan
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

print("Cleaning GTFS time formatting...")
red_line_df['scheduled_seconds'] = red_line_df['arrival_time'].apply(time_to_seconds)

# Drop any rows with missing schedule times
red_line_df = red_line_df.dropna(subset=['scheduled_seconds']).copy()


# ==========================================
# ADDING SYNTHETIC DELAY (Prerequisite for Task 2)
# ==========================================
def generate_synthetic_delay(row):
    # Base delay: Mostly 0-2 minutes (using an exponential distribution)
    delay_minutes = np.random.exponential(scale=1.0) 
    
    # 1. Peak Hour Penalty (8 AM - 10 AM, 5 PM - 8 PM)
    hour = int(row['arrival_time'].split(':')[0]) % 24
    if (8 <= hour <= 10) or (17 <= hour <= 20):
        delay_minutes += np.random.uniform(1, 5) # Add 1 to 5 extra minutes
        
    # 2. Busy Station Penalty (Major interchanges usually cause dwell-time delays)
    if row['stop_name'] in ['Ameerpet', 'MG Bus Station', 'Parade Ground']:
        delay_minutes += np.random.uniform(1, 3)
        
    # 3. On-time probability (15% chance a train is perfectly on time)
    if random.random() < 0.15:
        delay_minutes = 0
        
    return int(delay_minutes * 60) # Convert back to seconds

# Generate the realistic "Actual Arrival Time"
np.random.seed(42) # Ensures we get the same "random" delays every time we run this
red_line_df['actual_seconds'] = red_line_df['scheduled_seconds'] + red_line_df.apply(generate_synthetic_delay, axis=1)


# ==========================================
# TASK 2: Create Delay Feature
# ==========================================
# As per project plan: actual - scheduled time
print("Calculating target variable (Delay)...")
red_line_df['delay_seconds'] = red_line_df['actual_seconds'] - red_line_df['scheduled_seconds']

# Convert to minutes for easier understanding and modeling
red_line_df['delay_minutes'] = round(red_line_df['delay_seconds'] / 60, 2)


# ==========================================
# TASK 3: Feature Engineering
# ==========================================
print("Engineering features (hour, day, peak time, station)...")

# Extract the Hour
red_line_df['hour'] = (red_line_df['scheduled_seconds'] // 3600) % 24

# Create Peak Time Flag (1 for peak, 0 for off-peak)
red_line_df['is_peak_hour'] = red_line_df['hour'].apply(lambda x: 1 if (8 <= x <= 10) or (17 <= x <= 20) else 0)

# Simulate Days of the Week (Since static GTFS doesn't have dynamic dates, we assign them to build our ML dataset)
# 0 = Monday, 6 = Sunday
red_line_df['day_of_week'] = np.random.randint(0, 7, size=len(red_line_df))

# Create Weekend Flag
red_line_df['is_weekend'] = red_line_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# ==========================================
# FINAL CLEANUP & SAVING
# ==========================================
# Keep only the features we will feed into our Machine Learning models
ml_features = ['stop_name', 'hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'delay_minutes']
ml_ready_df = red_line_df[ml_features].copy()

# Save this processed dataset to our data/processed/ folder!
ml_ready_df.to_csv('../data/processed/red_line_ml_data.csv', index=False)

print("\n--- Phase 2 Complete ---")
print(f"Processed dataset saved with {len(ml_ready_df)} rows.")
print(ml_ready_df.head())

Cleaning GTFS time formatting...
Calculating target variable (Delay)...
Engineering features (hour, day, peak time, station)...

--- Phase 2 Complete ---
Processed dataset saved with 30648 rows.
             stop_name  hour  day_of_week  is_weekend  is_peak_hour  \
571      Dilsukh Nagar     6            4           0             0   
572      Chaitanyapuri     6            4           0             0   
573  Victoria Memorial     6            4           0             0   
574        L. B. Nagar     6            6           1             0   
575      Gandhi Bhavan     6            6           1             0   

     delay_minutes  
571           0.47  
572           3.00  
573           1.32  
574           0.90  
575           0.17  
